# StepManAI — Training

Trains the step **placement** and **selection** models on your DDR library features.

**Before running:** `Runtime → Change runtime type → T4 GPU`

The dataset (mel features + chart labels for ~1800 songs) downloads automatically
from the GitHub release. Checkpoints are saved to `MyDrive/StepManAI/checkpoints/`
(used by the generate notebook).

In [ ]:
#@title 1. Setup — mount Drive, fetch code + dataset
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/StepManAI'):
    !git clone -q https://github.com/Mrman67/StepManAI.git /content/StepManAI
REL = 'https://github.com/Mrman67/StepManAI/releases/download/dataset-v1'
if not os.path.exists('/content/data/labels.pkl'):
    print('downloading dataset from GitHub release...')
    for part in ['aa', 'ab', 'ac']:
        !wget -q --show-progress -O /content/ds.tar.{part} {REL}/stepmanai_dataset.tar.{part}
    !cat /content/ds.tar.* > /content/dataset.tar && rm /content/ds.tar.*
    !mkdir -p /content/data && tar -xf /content/dataset.tar -C /content/data && rm /content/dataset.tar
import glob
print(len(glob.glob('/content/data/cache_u8/*.npy')), 'songs ready')

In [ ]:
#@title 2. Train placement model (when steps happen)
epochs = 40 #@param {type:"integer"}
batch_size = 64 #@param {type:"integer"}
import os
os.environ['STEPMANAI_ROOT'] = '/content/StepManAI'
os.environ['STEPMANAI_CACHE'] = '/content/data/cache_u8'
os.environ['STEPMANAI_LABELS'] = '/content/data/labels.pkl'
os.environ['EPOCHS'] = str(epochs)
os.environ['BATCH'] = str(batch_size)
!cd /content/StepManAI && python train_placement.py

In [ ]:
#@title 3. Train selection model (which arrows)
epochs = 30 #@param {type:"integer"}
import os
os.environ['STEPMANAI_ROOT'] = '/content/StepManAI'
os.environ['STEPMANAI_LABELS'] = '/content/data/labels.pkl'
os.environ['EPOCHS'] = str(epochs)
!cd /content/StepManAI && python train_selection.py

In [ ]:
#@title 4. Save checkpoints to Drive
!mkdir -p /content/drive/MyDrive/StepManAI/checkpoints
!cp /content/StepManAI/checkpoints/placement.pt /content/drive/MyDrive/StepManAI/checkpoints/
!cp /content/StepManAI/checkpoints/selection.pt /content/drive/MyDrive/StepManAI/checkpoints/
!ls -la /content/drive/MyDrive/StepManAI/checkpoints/
print('done — open the generate notebook!')